# Module 04 — Notebook 03: Shapley Value Sampling

## Learning Objectives
By completing this notebook, you will:
1. Apply `ShapleyValueSampling` from Captum to tabular wine quality data
2. Assess convergence of Shapley estimates as a function of `n_samples`
3. Verify the Efficiency (completeness) property: sum(phi_i) = f(x) - f(baseline)
4. Compare Shapley values, Feature Ablation, and IG on the same predictions

## Prerequisites
- Module 04 Notebook 02 (Feature Ablation)
- Guide 02 (Shapley values theory)

## Estimated Time: 13 minutes

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import urllib.request
import io

from captum.attr import ShapleyValueSampling, FeatureAblation, IntegratedGradients

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")

FEATURE_NAMES = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid',
    'residual_sugar', 'chlorides', 'free_SO2',
    'total_SO2', 'density', 'pH', 'sulphates', 'alcohol'
]

In [ ]:
# Load and prepare data (same as Notebook 02)
WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

try:
    with urllib.request.urlopen(WINE_URL, timeout=15) as resp:
        df = pd.read_csv(io.BytesIO(resp.read()), sep=';')
except Exception:
    print("Using synthetic fallback dataset.")
    n = 1599
    df = pd.DataFrame({
        'fixed acidity':          np.random.uniform(4.6, 15.9, n),
        'volatile acidity':        np.random.uniform(0.12, 1.58, n),
        'citric acid':             np.random.uniform(0.0, 1.0, n),
        'residual sugar':          np.random.uniform(1.2, 15.5, n),
        'chlorides':               np.random.uniform(0.012, 0.611, n),
        'free sulfur dioxide':     np.random.uniform(1.0, 72.0, n),
        'total sulfur dioxide':    np.random.uniform(6.0, 289.0, n),
        'density':                 np.random.uniform(0.990, 1.004, n),
        'pH':                      np.random.uniform(2.74, 4.01, n),
        'sulphates':               np.random.uniform(0.33, 2.0, n),
        'alcohol':                 np.random.uniform(8.4, 14.9, n),
        'quality':                 np.random.choice(range(3, 9), n)
    })

X = df.drop('quality', axis=1).values.astype(np.float32)
y = (df['quality'].values >= 6).astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)
X_mean = torch.tensor(X_train.mean(axis=0), dtype=torch.float32).unsqueeze(0)


# Train MLP (identical to Notebook 02)
class WineQualityMLP(nn.Module):
    def __init__(self, n_features=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)

model = WineQualityMLP().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(80):
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t.long())
    optimizer.zero_grad(); loss.backward(); optimizer.step()

model.eval()
with torch.no_grad():
    test_probs = torch.softmax(model(X_test_t), dim=1)
    test_acc = (model(X_test_t).argmax(1) == y_test_t.long()).float().mean()
print(f"Model trained. Test accuracy: {test_acc:.1%}")

## 1. ShapleyValueSampling: Basic Application

We compute Shapley value attributions for the same confidently-good and confidently-poor wine samples from Notebook 02.

In [ ]:
svs = ShapleyValueSampling(model)

# Find the most confidently predicted samples
good_probs = test_probs[:, 1]
good_idx = good_probs.argmax().item()
poor_idx = good_probs.argmin().item()

good_wine = X_test_t[good_idx].unsqueeze(0)
poor_wine = X_test_t[poor_idx].unsqueeze(0)

N_SAMPLES = 200  # Monte Carlo samples for Shapley

print(f"Computing Shapley values with n_samples={N_SAMPLES}...")

# Shapley values for good wine
shap_good = svs.attribute(
    good_wine,
    baselines=X_mean,
    target=1,             # "Good wine" class
    n_samples=N_SAMPLES
)
shap_good_np = shap_good.squeeze().detach().numpy()

# Shapley values for poor wine
shap_poor = svs.attribute(
    poor_wine,
    baselines=X_mean,
    target=1,
    n_samples=N_SAMPLES
)
shap_poor_np = shap_poor.squeeze().detach().numpy()

print(f"Good wine — P(good)={good_probs[good_idx]:.1%}")
print(f"  Sum of Shapley values: {shap_good_np.sum():.5f}")

# Verify Efficiency axiom (completeness)
with torch.no_grad():
    f_good  = model(good_wine)[0, 1].item()
    f_base  = model(X_mean)[0, 1].item()
total_diff_good = f_good - f_base
print(f"  f(x) - f(baseline): {total_diff_good:.5f}")
print(f"  Efficiency error: {abs(shap_good_np.sum() - total_diff_good):.5f}")

## 2. Convergence Analysis: How Many Samples Are Needed?

Shapley Value Sampling is stochastic. We assess how the estimates stabilize as `n_samples` increases.

In [ ]:
# Sweep n_samples and measure convergence
torch.manual_seed(42)
np.random.seed(42)

n_samples_list = [5, 10, 25, 50, 100, 200]
attrs_by_n = []

print("Sweeping n_samples for Shapley convergence...")
for n in n_samples_list:
    attr = svs.attribute(
        good_wine, baselines=X_mean,
        target=1, n_samples=n
    )
    attrs_by_n.append(attr.squeeze().detach().numpy())
    print(f"  n_samples={n:4d}: sum={attr.squeeze().sum().item():+.5f}")

# Measure stability: maximum change from one n to the next
max_changes = []
for i in range(1, len(attrs_by_n)):
    change = np.abs(attrs_by_n[i] - attrs_by_n[i-1]).max()
    max_changes.append(change)

# Visualize convergence
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Shapley Value Sampling Convergence\n'
    f'(Good wine — P(good)={good_probs[good_idx]:.1%})',
    fontsize=12
)

# Attribution trajectories
attrs_array = np.stack(attrs_by_n)  # (n_n_samples, 11)
colors = plt.cm.tab10(np.linspace(0, 1, len(FEATURE_NAMES)))
for feat_idx, color in enumerate(colors):
    axes[0].plot(n_samples_list, attrs_array[:, feat_idx],
                  'o-', color=color, linewidth=1.5, markersize=5,
                  label=FEATURE_NAMES[feat_idx][:8], alpha=0.8)
axes[0].set_xlabel('n_samples')
axes[0].set_ylabel('Shapley Attribution')
axes[0].set_title('Attribution Trajectories vs n_samples')
axes[0].legend(fontsize=7, ncol=2, loc='upper right')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].grid(True, alpha=0.3)

# Max change between successive estimates
axes[1].plot(n_samples_list[1:], max_changes, 'o-',
              color='darkorange', linewidth=2, markersize=8)
axes[1].axhline(0.01, color='green', linestyle='--', label='0.01 (good)')
axes[1].axhline(0.05, color='red',   linestyle='--', label='0.05 (acceptable)')
axes[1].set_xlabel('n_samples')
axes[1].set_ylabel('Max |attribution change|')
axes[1].set_title('Convergence: Max Change per n_samples Doubling')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Final attributions (highest n_samples)
final_attrs = attrs_by_n[-1]
sorted_idx = np.argsort(final_attrs)
colors_bar = ['#2ecc71' if v >= 0 else '#e74c3c' for v in final_attrs[sorted_idx]]
axes[2].barh([FEATURE_NAMES[i] for i in sorted_idx],
              final_attrs[sorted_idx],
              color=colors_bar, alpha=0.85)
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_xlabel('Shapley Attribution')
axes[2].set_title(f'Final Attributions (n_samples={n_samples_list[-1]})')

plt.tight_layout()
plt.show()

# Find convergence point
for i, (n, change) in enumerate(zip(n_samples_list[1:], max_changes)):
    if change < 0.01:
        print(f"Converged at n_samples={n} (max change={change:.5f} < 0.01)")
        break
else:
    print(f"Did not fully converge by n_samples={n_samples_list[-1]}")
    print(f"(Final max change: {max_changes[-1]:.5f})")

## 3. Three-Method Comparison: Shapley vs FA vs IG

We compare all three attribution methods on the same predictions. High agreement across methods = robust finding.

In [ ]:
fa  = FeatureAblation(model)
ig  = IntegratedGradients(model)

# Compute all three for good wine
shap_attr = shap_good_np  # Already computed

fa_attr = fa.attribute(
    good_wine, baselines=X_mean, target=1
).squeeze().detach().numpy()

ig_attr, ig_delta = ig.attribute(
    good_wine.clone().requires_grad_(True),
    baselines=X_mean, target=1,
    n_steps=100, return_convergence_delta=True
)
ig_attr_np = ig_attr.squeeze().detach().numpy()
print(f"IG convergence delta: {ig_delta.item():.5f}")

# Normalize each method's attribution to [-1, 1] for fair comparison
def normalize_attrs(arr):
    max_abs = np.abs(arr).max() + 1e-8
    return arr / max_abs

shap_norm = normalize_attrs(shap_attr)
fa_norm   = normalize_attrs(fa_attr)
ig_norm   = normalize_attrs(ig_attr_np)

# Pairwise Spearman correlations
rho_shap_fa, _ = spearmanr(shap_norm, fa_norm)
rho_shap_ig, _ = spearmanr(shap_norm, ig_norm)
rho_fa_ig,   _ = spearmanr(fa_norm, ig_norm)

print(f"\nPairwise rank correlations:")
print(f"  Shapley vs FA:  ρ = {rho_shap_fa:.4f}")
print(f"  Shapley vs IG:  ρ = {rho_shap_ig:.4f}")
print(f"  FA vs IG:       ρ = {rho_fa_ig:.4f}")

In [ ]:
# Three-method comparison visualization
x = np.arange(len(FEATURE_NAMES))
w = 0.27

# Sort by Shapley magnitude
sort_idx = np.argsort(np.abs(shap_norm))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    f'Three-Method Attribution Comparison — Confidently Good Wine\n'
    f'P(good)={good_probs[good_idx]:.1%}',
    fontsize=12
)

# Bar chart: all three methods side by side
pos = np.arange(len(FEATURE_NAMES))
axes[0].bar(pos - w, shap_norm[sort_idx], w,
             label='Shapley Values', color='#9b59b6', alpha=0.85)
axes[0].bar(pos,     fa_norm[sort_idx],   w,
             label='Feature Ablation', color='#e74c3c', alpha=0.85)
axes[0].bar(pos + w, ig_norm[sort_idx],   w,
             label='Integrated Gradients', color='#3498db', alpha=0.85)
axes[0].set_xticks(pos)
axes[0].set_xticklabels(
    [FEATURE_NAMES[i][:10] for i in sort_idx],
    rotation=45, ha='right'
)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_ylabel('Normalized Attribution ([-1, 1])')
axes[0].set_title(f'Shapley vs FA vs IG\nSorted by |Shapley| rank')
axes[0].legend(fontsize=10)

# Pairwise scatter matrix (3 pairs)
axes[1].scatter(ig_norm, shap_norm,
                 s=100, c='#9b59b6', label=f'IG vs Shapley (ρ={rho_shap_ig:.3f})',
                 alpha=0.8, zorder=3)
axes[1].scatter(fa_norm, shap_norm,
                 s=100, c='#e74c3c', label=f'FA vs Shapley (ρ={rho_shap_fa:.3f})',
                 marker='^', alpha=0.8, zorder=2)
axes[1].scatter(ig_norm, fa_norm,
                 s=100, c='#3498db', label=f'IG vs FA (ρ={rho_fa_ig:.3f})',
                 marker='s', alpha=0.8, zorder=1)
diag = np.linspace(-1, 1, 100)
axes[1].plot(diag, diag, 'k--', alpha=0.3, lw=1, label='Perfect agreement')
axes[1].set_xlabel('Method A Attribution')
axes[1].set_ylabel('Method B Attribution')
axes[1].set_title('Pairwise Method Agreement')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("High ρ across all pairs → robust attribution (methods agree)")
print("Low ρ for Shapley vs IG → feature interactions present (Shapley captures them, IG doesn't)")

## 4. Exercise: Efficiency Verification Across Samples

Verify the Efficiency (completeness) property of Shapley values for 20 test samples. For each:
- Compute Shapley attributions
- Compute f(x) - f(baseline)
- Verify: |sum(phi_i) - (f(x) - f(baseline))| < tolerance

In [ ]:
# Efficiency (completeness) verification across 20 samples
N_VERIFY = 20
TOLERANCE = 0.05  # Acceptable approximation error

efficiency_errors = []
shapley_sums = []
f_diffs = []

with torch.no_grad():
    f_baseline = model(X_mean)[0, 1].item()

print(f"Verifying Efficiency axiom for {N_VERIFY} samples...")
print(f"f(baseline) = {f_baseline:.5f}")
print()

for i in range(N_VERIFY):
    sample = X_test_t[i].unsqueeze(0)
    target = 1  # Always explain good wine class for consistency

    attr = svs.attribute(
        sample, baselines=X_mean,
        target=target, n_samples=100
    )
    attr_sum = attr.squeeze().sum().item()

    with torch.no_grad():
        f_x = model(sample)[0, 1].item()
    f_diff = f_x - f_baseline

    error = abs(attr_sum - f_diff)
    shapley_sums.append(attr_sum)
    f_diffs.append(f_diff)
    efficiency_errors.append(error)

    status = 'OK' if error < TOLERANCE else 'FAIL'
    print(f"Sample {i:3d}: sum(phi)={attr_sum:+.5f}  "
          f"f(x)-f(bl)={f_diff:+.5f}  error={error:.5f} [{status}]")

# Summary
n_pass = sum(1 for e in efficiency_errors if e < TOLERANCE)
print(f"\n{'='*50}")
print(f"Efficiency verified: {n_pass}/{N_VERIFY} samples within tolerance={TOLERANCE}")
print(f"Mean error: {np.mean(efficiency_errors):.5f}")
print(f"Max error:  {np.max(efficiency_errors):.5f}")

In [ ]:
# Visualize efficiency verification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Shapley Value Sampling: Efficiency (Completeness) Verification', fontsize=12)

# Scatter: sum(phi) vs f(x) - f(baseline)
axes[0].scatter(f_diffs, shapley_sums, c='navy', s=60, alpha=0.8, zorder=2)
diag = np.linspace(min(f_diffs + shapley_sums) - 0.01,
                    max(f_diffs + shapley_sums) + 0.01, 100)
axes[0].plot(diag, diag, 'r--', lw=1.5, label='Perfect efficiency')
axes[0].set_xlabel('f(x) - f(baseline)  (theoretical)')
axes[0].set_ylabel('sum(Shapley values)  (actual)')
axes[0].set_title(f'Efficiency Verification\n(n_samples=100)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram of errors
axes[1].hist(efficiency_errors, bins=15, color='darkorange', alpha=0.85,
              edgecolor='none')
axes[1].axvline(TOLERANCE, color='red', linestyle='--',
                  label=f'Tolerance = {TOLERANCE}')
axes[1].axvline(np.mean(efficiency_errors), color='green', linestyle='--',
                  label=f'Mean = {np.mean(efficiency_errors):.4f}')
axes[1].set_xlabel('|sum(phi) - (f(x)-f(baseline))|')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Efficiency Errors')
axes[1].legend()

plt.tight_layout()
plt.show()

# Validation assertions
assert n_pass >= N_VERIFY * 0.8, \
    f"Too many samples failed efficiency check: {N_VERIFY - n_pass}/{N_VERIFY}"
print(f"Validation passed: {n_pass}/{N_VERIFY} samples within tolerance.")

## Summary

### What You Learned

1. **ShapleyValueSampling** approximates Shapley values via Monte Carlo averaging over random feature orderings
2. **Convergence:** attributions stabilize with increasing `n_samples`; typically n_samples=100-200 is sufficient for tabular data
3. **Efficiency axiom** holds (within numerical approximation): sum(phi_i) ≈ f(x) - f(baseline)
4. **Agreement with FA and IG** indicates robust feature attributions; disagreement highlights potential feature interactions
5. **Shapley captures interactions** — when features interact (e.g., alcohol matters more in high-sulphate wines), Shapley correctly attributes shared credit

### Key API
```python
from captum.attr import ShapleyValueSampling

svs = ShapleyValueSampling(model)
attr = svs.attribute(
    input_tensor,       # (1, n_features)
    baselines=baseline, # Training mean
    target=class_idx,
    n_samples=200       # More samples = lower variance
)
# attr: (1, n_features)
# attr.sum() ≈ f(x) - f(baseline)  [Efficiency axiom]
```

### Module 04 Complete

You now have a complete perturbation attribution toolkit:
- **Occlusion:** spatial heatmaps for images via sliding window
- **Feature Ablation:** flexible feature group attribution with superpixels or tabular features
- **Shapley Value Sampling:** game-theoretically fair attribution with interaction effects

**Quick-starts** are next: applying everything you've learned to image, text, and tabular attribution in under 2 minutes each.